# 03 — Run agent variant (Colab T4)

Run **`midi-agent run`**. **Runtime → T4 GPU** for Llama variants.

- **T4:** use **`Llama-3.2-1B-Instruct`** (8B often OOMs).
- **MCP (g3):** set `USE_MCP=True` or `VARIANT="g3"` — installs [music21-mcp](https://github.com/cclawton/music21-mcp) and passes `--mcp-command` (perception only; edits still use `execute_plan`).

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
MIDI_AGENT_REPO = "https://github.com/juliagsy/midi-agent.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
MIDI_AGENT = f"{WORK}/midi-agent"

DATASET_MODE = "real_v0.2"  # @param ["pilot", "real_v0.2"]
RUN_ID = "real_v0.2_v1" if DATASET_MODE == "real_v0.2" else "pilot_v1"  # @param {type:"string"}
VARIANT = "g1"  # @param ["oracle", "g1", "g2", "g3", "g4"]
AGENT_RUN_SUFFIX = ""  # @param {type:"string"}
MODEL = "meta-llama/Llama-3.2-1B-Instruct"  # @param {type:"string"}
DEVICE = "cuda"  # @param ["cuda", "cpu", "auto"]
SPLIT = "test"  # @param ["test", "validation"]
MAX_ITEMS = None  # @param {type:"raw"}  smoke: 5
MAX_ROUNDS = 3  # @param {type:"integer"}
MAX_NEW_TOKENS = 256  # @param {type:"integer"}
RESUME = True  # @param {type:"boolean"}

USE_MCP = False  # @param {type:"boolean"}  auto-on for VARIANT=g3; optional on g1/g2/g4
MUSIC21_MCP_REPO = "https://github.com/cclawton/music21-mcp.git"  # @param {type:"string"}
MCP_COMMAND = "python"  # @param {type:"string"}
MCP_ARGS = ["-m", "music21_mcp.server"]  # stdio server module

LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
MANIFEST = f"{LOCAL_DATA}/{MANIFEST_NAME}"
RUN_FOLDER = f"agent_{VARIANT}" + (f"_{AGENT_RUN_SUFFIX}" if AGENT_RUN_SUFFIX else "")
if USE_MCP or VARIANT == "g3":
    RUN_FOLDER = RUN_FOLDER + "_mcp"

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_MI = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_MI}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_MI}/data/v0.2"
DRIVE_AGENT = "/content/drive/MyDrive/midi-agent"
DRIVE_RESULTS = f"{DRIVE_AGENT}/runs/{RUN_ID}/{RUN_FOLDER}"
LOCAL_RESULTS = f"{WORK}/runs/{RUN_FOLDER}"

ENABLE_MCP = USE_MCP or VARIANT == "g3"
if VARIANT == "g3" and not USE_MCP:
    print("VARIANT=g3 → MCP enabled (set USE_MCP=False only with --mcp-command added manually)")

In [ ]:
# @title Bootstrap
import os
import shutil
import subprocess
from getpass import getpass
from pathlib import Path

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

if VARIANT != "oracle" and DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available — Runtime → T4 GPU, or DEVICE='cpu'")
    print("GPU:", torch.cuda.get_device_name(0))

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if VARIANT != "oracle":
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

for path, url in [(MUSICINSTRUCT, MUSICINSTRUCT_REPO), (MIDI_AGENT, MIDI_AGENT_REPO)]:
    if Path(path).is_dir():
        shutil.rmtree(path)
    sh(f"git clone --branch {BRANCH} --depth 1 {url} {path}")

extras = "llm,mcp" if ENABLE_MCP else "llm"
sh("pip install -q -U transformers accelerate huggingface_hub")
sh(f"pip install -q -e {MUSICINSTRUCT}")
if ENABLE_MCP:
    sh(f"pip install -q 'git+{MUSIC21_MCP_REPO}'")
sh(f"pip install -q -e '{MIDI_AGENT}[{extras}]'")

if ENABLE_MCP:
    import importlib.util
    if importlib.util.find_spec("music21_mcp.server") is None:
        raise RuntimeError("music21-mcp install failed — check MUSIC21_MCP_REPO")
    print("MCP server:", MCP_COMMAND, " ".join(MCP_ARGS))

Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
Path(LOCAL_RESULTS).mkdir(parents=True, exist_ok=True)
if USE_DRIVE and Path(DRIVE_RESULTS).is_dir():
    for name in ("predictions.jsonl", "agent_report.json"):
        src = Path(DRIVE_RESULTS) / name
        if src.is_file():
            shutil.copy2(src, Path(LOCAL_RESULTS) / name)
    print("Restored from", DRIVE_RESULTS)

In [ ]:
# @title Run midi-agent
import json
import os
import subprocess
import sys
from pathlib import Path

cmd = [
    "midi-agent", "run", MANIFEST,
    "--variant", VARIANT,
    "--output-dir", LOCAL_RESULTS,
    "--split", SPLIT,
    "--max-rounds", str(MAX_ROUNDS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--device", DEVICE,
]
if VARIANT != "oracle":
    cmd.extend(["--model", MODEL])
if ENABLE_MCP:
    cmd.append("--mcp-command")
    cmd.append(MCP_COMMAND)
    for arg in MCP_ARGS:
        cmd.extend(["--mcp-arg", arg])
if MAX_ITEMS:
    cmd.extend(["--max-items", str(int(MAX_ITEMS))])
if RESUME:
    cmd.append("--resume")

env = os.environ.copy()
print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=MIDI_AGENT, env=env, capture_output=True, text=True)
if proc.stdout:
    print(proc.stdout)
if proc.returncode != 0:
    print("--- stderr ---", file=sys.stderr)
    print(proc.stderr or "(empty)", file=sys.stderr)
    pred = Path(LOCAL_RESULTS) / "predictions.jsonl"
    report = Path(LOCAL_RESULTS) / "agent_report.json"
    if pred.is_file():
        print(f"Partial predictions: {pred} ({sum(1 for _ in pred.open())} lines)")
    if report.is_file():
        print("Last report:", report.read_text()[:4000])
    proc.check_returncode()

print(json.dumps(json.loads(Path(LOCAL_RESULTS).joinpath("agent_report.json").read_text()), indent=2))

In [ ]:
# @title Sync to Drive
import shutil
from pathlib import Path

if USE_DRIVE:
    Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)
    for name in Path(LOCAL_RESULTS).iterdir():
        if name.is_file():
            shutil.copy2(name, Path(DRIVE_RESULTS) / name.name)
        elif name.is_dir() and name.name == "midi":
            dest = Path(DRIVE_RESULTS) / "midi"
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(name, dest)
    print("Saved to", DRIVE_RESULTS)